In [1]:
# Import

import os
import random
import numpy as np
import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import DataCollatorWithPadding

In [2]:
# Setup, reproducibility, device
SEED = 123

def set_all_seeds(seed: int = SEED):
    # Fix all RNG sources: high variance with ~137 training docs, repeatability matters
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # no-op if no CUDA

set_all_seeds(SEED)

# Deterministic cuDNN: slightly slower, reproducible results (negligible cost on small data)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Auto device detection: CUDA -> MPS -> CPU
def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[device] CUDA active -> {torch.cuda.get_device_name(0)}")
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("[device] Apple MPS active")
    else:
        dev = torch.device("cpu")
        print("[device] No GPU found -> CPU (BERT will be slow)")
    return dev

device = get_device()



[device] CUDA active -> NVIDIA GeForce RTX 5070


In [3]:
# Sanity check: 'CUDA build=None' means a CPU-only torch is installed
print(f"[versions] torch={torch.__version__} | CUDA build={torch.version.cuda}")
print(f"[seed] {SEED}")

[versions] torch=2.12.1+cu130 | CUDA build=13.0
[seed] 123


In [4]:
# Load data, select columns, integrity checks, token-length profile

CSV_PATH = "..\Dati\Processed\dataset_processed_quantile1_sentences.csv"  
MODEL_NAME = "bert-base-cased"                            # cased: casing/punct are stylistic signal
KEEP_COLS = ["article_id", "topic_id", "binary_label", "fold", "text_bert"]

# Load 
df = pd.read_csv(CSV_PATH)
df = df[KEEP_COLS].copy()
df = df.reset_index(drop=True)  # avoid __index_level_0__ leaking into HF Dataset later

# Integrity checks 
print(f"[shape] {df.shape[0]} docs, {df.shape[1]} cols")
print(f"[NaN]\n{df.isna().sum()}")
print(f"[dup article_id] {df['article_id'].duplicated().sum()}")
print(f"[folds] {sorted(df['fold'].unique())}")

# Class balance overall and per fold (must stay ~2:1)
print("\n[label balance overall]")
print(df["binary_label"].value_counts(normalize=True).round(3).to_dict())
print("\n[label balance per fold]")
print(df.groupby("fold")["binary_label"].value_counts(normalize=True).round(2).unstack())
print("\n[docs per fold]")
print(df["fold"].value_counts().sort_index().to_dict())



# Token-length profile (does max_length=512 actually bite?) 
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
# add_special_tokens=True counts [CLS]/[SEP] as in real training
tok_lens = df["text_bert"].apply(lambda t: len(tok(t, add_special_tokens=True)["input_ids"]))
df["_tok_len"] = tok_lens

print("\n[token length] describe")
print(tok_lens.describe().round(1))
for thr in (128, 256, 384, 512):
    share = (tok_lens > thr).mean()
    print(f"  > {thr} tokens: {share:.1%} of docs")

[shape] 171 docs, 5 cols
[NaN]
article_id      0
topic_id        0
binary_label    0
fold            0
text_bert       0
dtype: int64
[dup article_id] 0
[folds] [0, 1, 2, 3, 4]

[label balance overall]
{1: 0.667, 0: 0.333}

[label balance per fold]
binary_label     0     1
fold                    
0             0.33  0.67
1             0.33  0.67
2             0.33  0.67
3             0.33  0.67
4             0.33  0.67

[docs per fold]
{0: 36, 1: 36, 2: 33, 3: 33, 4: 33}


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (981 > 512). Running this sequence through the model will result in indexing errors



[token length] describe
count     171.0
mean      286.0
std       262.8
min        36.0
25%       144.0
50%       211.0
75%       329.5
max      1795.0
Name: text_bert, dtype: float64
  > 128 tokens: 79.5% of docs
  > 256 tokens: 39.2% of docs
  > 384 tokens: 18.7% of docs
  > 512 tokens: 10.5% of docs


In [5]:
# How many articles are above 512 tokens?
over = df.loc[df["_tok_len"] > 512, "_tok_len"]
print(over.describe().round(0))
print("quantili:", over.quantile([.5, .75, .9, .95]).round(0).to_dict())

#Perdiamo abbastanz ainformazione su 18 articoli, ma le altre opzioni sono esegeratamente
#lunghe e complesse, secondo me accettabile

count      18.0
mean      888.0
std       384.0
min       517.0
25%       605.0
50%       778.0
75%       986.0
max      1795.0
Name: _tok_len, dtype: float64
quantili: {0.5: 778.0, 0.75: 986.0, 0.9: 1394.0, 0.95: 1684.0}


In [6]:
# Are the truncated (long) docs concentrated in one class?
long_mask = df["_tok_len"] > 512
print(df.loc[long_mask, "binary_label"].value_counts().to_dict())
print("share of long docs that are class 1:",
      df.loc[long_mask, "binary_label"].mean().round(2))
print("baseline share class 1 overall:", df["binary_label"].mean().round(2))

{1: 14, 0: 4}
share of long docs that are class 1: 0.78
baseline share class 1 overall: 0.67


In [7]:
# Tokenization, dynamic-padding collator

MAX_LEN = 512  # 10% of docs exceed this; head-only truncation (bias concentrated in head)

# Build HF dataset; keep 'fold' for splitting and 'article_id' for OOF tracking
base_cols = ["text_bert", "binary_label", "fold", "article_id"]
ds_full = Dataset.from_pandas(df[base_cols].reset_index(drop=True), preserve_index=False)

def tokenize_batch(batch):
    return tok(batch["text_bert"], truncation=True, max_length=MAX_LEN)


In [8]:
ds_full = ds_full.map(tokenize_batch, batched=True)
ds_full = ds_full.rename_column("binary_label", "labels")

# Dynamic per-batch padding (not fixed 512) -> less wasted compute/memory
collator = DataCollatorWithPadding(tokenizer=tok)

# Columns the model's forward actually consumes; everything else gets stripped
MODEL_COLS = ["input_ids", "attention_mask", "token_type_ids", "labels"]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

In [9]:
# Per-fold split builder

def make_fold(k):
    """Split for fold k. Returns cleaned train/eval datasets (tensor cols only),
    plus eval article_ids and gold labels (in eval order) for OOF tracking."""
    folds = ds_full["fold"]
    train_idx = [i for i, f in enumerate(folds) if f != k]
    eval_idx  = [i for i, f in enumerate(folds) if f == k]

    train_ds = ds_full.select(train_idx)
    eval_ds  = ds_full.select(eval_idx)

    # Capture identity BEFORE dropping string columns
    eval_ids    = list(eval_ds["article_id"])
    eval_labels = list(eval_ds["labels"])

    # Strip non-model columns (a string col would break the padding collator)
    drop = [c for c in train_ds.column_names if c not in MODEL_COLS]
    train_ds = train_ds.remove_columns(drop)
    eval_ds  = eval_ds.remove_columns(drop)

    return train_ds, eval_ds, eval_ids, eval_labels

In [10]:
# Sanity check on one fold 
tr, ev, ids, y = make_fold(0)
print(f"[fold 0] train={len(tr)}  eval={len(ev)}")
print(f"[fold 0] model cols = {tr.column_names}")
print(f"[fold 0] eval class balance = "
      f"{pd.Series(y).value_counts(normalize=True).round(2).to_dict()}")

[fold 0] train=135  eval=36
[fold 0] model cols = ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
[fold 0] eval class balance = {1: 0.67, 0: 0.33}
